# DA-Fusion Pipeline

Compares three data augmentation strategies using the same model and training setup as the CEMS pipeline:
- **Baseline**: Real data only, DINOv2 features from cache
- **RandAugment**: Real data with classical augmentation applied at DINOv2 extraction time
- **DA-Fusion**: Real + diffusion-generated synthetic images (from `generate_augmented.py`)

All experiments use `BiomassModel` (DINOv2 → 32-d latent → 5 targets) trained with `train_cems`.

## Imports and config

In [1]:
import sys, os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from sklearn.preprocessing import MinMaxScaler

REPO_ROOT = Path(r"C:\Users\Eier\OneDrive\Maskinlæring\INF367A\CSIRO---Image2Biomass-Prediction")
sys.path.insert(0, str(REPO_ROOT / "src"))

from shared.data_utils import make_splits, _make_wide_df, _image_id_from_path, TARGETS
from shared.model import BiomassModel
from shared.train import eval_epoch
from shared.metrics import weighted_global_r2, rmse_per_target
from methods.cems.train_cems import train_cems
from methods.cems.cems_utils import compute_neigh_size, precompute_knn
from methods.cems.estimate_id import estimate_id

CACHE_DIR = REPO_ROOT / "src" / "cache"
AUG_CSV   = REPO_ROOT / "data" / "tabular" / "train" / "train_augmented.csv"
CSV_PATH  = REPO_ROOT / "data" / "tabular" / "train" / "train.csv"
IMAGE_DIR = REPO_ROOT / "data" / "image"

cfg = dict(
    seed=42, epochs=80, lr=3e-4, weight_decay=1e-3,
    latent_dim=32, dropout=0.1, sigma=1e-3, cems_method=1,
)

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [2]:
import inspect
from shared.data_utils import make_splits
print(inspect.signature(make_splits))

train_ids, val_ids = make_splits(CSV_PATH)
print(f"Train: {len(train_ids)}  Val: {len(val_ids)}")

(csv_path=WindowsPath('C:/Users/Eier/OneDrive/Maskinlæring/INF367A/CSIRO---Image2Biomass-Prediction/data/tabular/train/train.csv'), n_splits=5, val_fold=0, group_by='visit')
Train: 286  Val: 71


## Load cached features and build splits

Reuses the DINOv2 cache from the CEMS pipeline. The same `GroupKFold` split is used across all experiments so results are directly comparable.

In [3]:
# --- Real training images (from CEMS cache) ---
features_np = np.load(CACHE_DIR / "features_dinov2.npy").astype(np.float32)   # (N, 384)
image_ids   = np.load(CACHE_DIR / "image_ids.npy", allow_pickle=True)          # (N,)

df_wide = _make_wide_df(CSV_PATH)
id_to_label = {
    _image_id_from_path(row["image_path"]): row[TARGETS].values.astype(np.float32)
    for _, row in df_wide.iterrows()
}
labels_np = np.stack(
    [id_to_label.get(iid, np.zeros(5, dtype=np.float32)) for iid in image_ids]
)  # (N, 5)

id_to_idx = {iid: i for i, iid in enumerate(image_ids)}
train_ids, val_ids = make_splits(CSV_PATH)

train_idx = [id_to_idx[i] for i in train_ids if i in id_to_idx]
val_idx   = [id_to_idx[i] for i in val_ids   if i in id_to_idx]

X_real_train = features_np[train_idx]   # (N_train, 384)
Y_real_train = labels_np[train_idx]     # (N_train, 5)
X_real_val   = features_np[val_idx]     # (N_val, 384)
Y_real_val   = labels_np[val_idx]       # (N_val, 5)

# Validation dataset — shared across all experiments
val_ds = TensorDataset(
    torch.tensor(X_real_val), torch.tensor(Y_real_val)
)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"N_train={len(train_idx)}  N_val={len(val_idx)}")
print(f"Real features: {X_real_train.shape}  Labels: {Y_real_train.shape}")

# --- Augmented (synthetic) images ---
AUG_CSV_EXISTS   = AUG_CSV.exists()
AUG_CACHE_EXISTS = (
    (CACHE_DIR / "augmented_features_dinov2.npy").exists() and
    (CACHE_DIR / "augmented_image_ids.npy").exists()
)

if AUG_CACHE_EXISTS and AUG_CSV_EXISTS:
    aug_features_np = np.load(CACHE_DIR / "augmented_features_dinov2.npy").astype(np.float32)
    aug_image_ids   = np.load(CACHE_DIR / "augmented_image_ids.npy", allow_pickle=True)

    df_aug = pd.read_csv(AUG_CSV)
    df_synth = df_aug[df_aug["is_synthetic"].astype(bool)].copy()
    # Map stem (e.g. 'ID123_aug0') → label row
    id_to_label_aug = {
        Path(row["image_path"]).stem: row[TARGETS].values.astype(np.float32)
        for _, row in df_synth.iterrows()
    }
    aug_labels_np = np.stack(
        [id_to_label_aug.get(iid, np.zeros(5, dtype=np.float32)) for iid in aug_image_ids]
    )  # (K, 5)

    print(f"Synthetic features: {aug_features_np.shape}  Labels: {aug_labels_np.shape}")
else:
    aug_features_np = None
    aug_image_ids   = None
    aug_labels_np   = None
    if not AUG_CSV_EXISTS:
        print("[WARN] train_augmented.csv not found. Run generate_augmented.py first.")
    if not AUG_CACHE_EXISTS:
        print("[WARN] Augmented DINOv2 cache not found. Run extract_features.py first.")

N_train=286  N_val=71
Real features: (286, 384)  Labels: (286, 5)
Synthetic features: (1071, 384)  Labels: (1071, 5)


In [4]:
# ===== LEAKAGE DIAGNOSTICS =====
# Runs three sanity checks on the baseline setup. Only the BiomassModel + ERM
# baseline is tested — if leakage exists here, it affects every experiment.

from shared.data_utils import time_split
from shared.train import train as train_erm
from sklearn.model_selection import GroupKFold, KFold
import pandas as pd

def _run_one(X_tr, Y_tr, X_va, Y_va, label):
    """Train a fresh BiomassModel and return val R²."""
    tr_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                          torch.tensor(Y_tr, dtype=torch.float32))
    va_ds = TensorDataset(torch.tensor(X_va, dtype=torch.float32),
                          torch.tensor(Y_va, dtype=torch.float32))
    torch.manual_seed(cfg["seed"])
    m = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)
    train_erm(m, tr_ds, va_ds,
              epochs=cfg["epochs"], lr=cfg["lr"],
              weight_decay=cfg["weight_decay"], seed=cfg["seed"],
              device=device, verbose=False)
    va_loader = DataLoader(va_ds, batch_size=64, shuffle=False)
    _, r2, _, _, _ = eval_epoch(m, va_loader, device)
    print(f"  {label:<40}  R² = {r2:.4f}")
    return r2

# ----- Test 1: 5-fold cross-validation with current GroupKFold -----
print("\n=== TEST 1: All 5 GroupKFold folds (current setup) ===")
df_wide_full = _make_wide_df(CSV_PATH)
all_ids = df_wide_full["image_path"].apply(_image_id_from_path).values

gkf = GroupKFold(n_splits=5)
r2_folds = []
for fold_i, (tr_idx, va_idx) in enumerate(gkf.split(df_wide_full,
                                                     groups=df_wide_full["image_path"])):
    tr_ids = set(all_ids[tr_idx])
    va_ids = set(all_ids[va_idx])
    tr_mask = np.array([iid in tr_ids for iid in image_ids])
    va_mask = np.array([iid in va_ids for iid in image_ids])
    r2_folds.append(_run_one(features_np[tr_mask], labels_np[tr_mask],
                             features_np[va_mask], labels_np[va_mask],
                             f"Fold {fold_i}"))
print(f"  Mean R² across folds: {np.mean(r2_folds):.4f}  ±  {np.std(r2_folds):.4f}")

# ----- Test 2: Time-based split (train on early dates, validate on late) -----
print("\n=== TEST 2: Time-based split (20% most recent as val) ===")
tr_df, va_df = time_split(df_wide_full, val_frac=0.2)
tr_ids = set(tr_df["image_path"].apply(_image_id_from_path))
va_ids = set(va_df["image_path"].apply(_image_id_from_path))
tr_mask = np.array([iid in tr_ids for iid in image_ids])
va_mask = np.array([iid in va_ids for iid in image_ids])
_run_one(features_np[tr_mask], labels_np[tr_mask],
         features_np[va_mask], labels_np[va_mask],
         "Time split (late = val)")

# ----- Test 3: Group by (State, Species, Sampling_Date) ~ paddock proxy -----
print("\n=== TEST 3: Group by (State, Species, Sampling_Date) — paddock proxy ===")
df_wide_full["group_key"] = (
    df_wide_full["State"].astype(str) + "|" +
    df_wide_full["Species"].astype(str) + "|" +
    df_wide_full["Sampling_Date"].astype(str)
)
print(f"  Unique groups: {df_wide_full['group_key'].nunique()} "
      f"(vs {len(df_wide_full)} images)")

gkf = GroupKFold(n_splits=5)
r2_folds_proxy = []
for fold_i, (tr_idx, va_idx) in enumerate(gkf.split(df_wide_full,
                                                     groups=df_wide_full["group_key"])):
    tr_ids = set(all_ids[tr_idx])
    va_ids = set(all_ids[va_idx])
    tr_mask = np.array([iid in tr_ids for iid in image_ids])
    va_mask = np.array([iid in va_ids for iid in image_ids])
    r2_folds_proxy.append(_run_one(features_np[tr_mask], labels_np[tr_mask],
                                    features_np[va_mask], labels_np[va_mask],
                                    f"Fold {fold_i} (paddock proxy)"))
print(f"  Mean R² across folds: {np.mean(r2_folds_proxy):.4f}  ±  {np.std(r2_folds_proxy):.4f}")

# ----- Summary -----
print("\n=== SUMMARY ===")
print(f"  Current GroupKFold (per-image):   {np.mean(r2_folds):.4f} ± {np.std(r2_folds):.4f}")
print(f"  Time split (late = val):          — see above")
print(f"  Paddock proxy (state+species+date): {np.mean(r2_folds_proxy):.4f} ± {np.std(r2_folds_proxy):.4f}")


=== TEST 1: All 5 GroupKFold folds (current setup) ===
  Fold 0                                    R² = 0.6799
  Fold 1                                    R² = 0.7348
  Fold 2                                    R² = 0.6158
  Fold 3                                    R² = 0.6313
  Fold 4                                    R² = 0.6856
  Mean R² across folds: 0.6695  ±  0.0423

=== TEST 2: Time-based split (20% most recent as val) ===
  Time split (late = val)                   R² = 0.5750

=== TEST 3: Group by (State, Species, Sampling_Date) — paddock proxy ===
  Unique groups: 40 (vs 357 images)
  Fold 0 (paddock proxy)                    R² = 0.6815
  Fold 1 (paddock proxy)                    R² = 0.6512
  Fold 2 (paddock proxy)                    R² = 0.5092
  Fold 3 (paddock proxy)                    R² = 0.5239
  Fold 4 (paddock proxy)                    R² = 0.5604
  Mean R² across folds: 0.5852  ±  0.0690

=== SUMMARY ===
  Current GroupKFold (per-image):   0.6695 ± 0.0423
  Time

## CEMS setup helper

Shared across all experiments: estimates latent ID, computes neighbourhood size, precomputes kNN.

In [5]:
def make_cems_args(X_train, Y_train, cfg, device, label=""):
    """
    Fit a temporary BiomassModel, estimate latent ID, precompute kNN.
    Returns (cems_args, knn_indices, neigh_size, y_scaler).
    """
    torch.manual_seed(cfg["seed"])
    _tmp = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)
    _tmp.eval()
    with torch.no_grad():
        latents = _tmp.encode(
            torch.tensor(X_train, dtype=torch.float32, device=device)
        ).cpu().numpy()
    d_float = estimate_id(latents, label=f"latent {label}")
    d = max(1, int(round(d_float)))
    neigh_size = compute_neigh_size(d)
    print(f"  {label}: d={d_float:.2f} → d={d}  neigh_size={neigh_size}")

    y_scaler = MinMaxScaler()
    y_scaler.fit(Y_train)
    Y_scaled = y_scaler.transform(Y_train).astype(np.float32)

    knn_indices = precompute_knn(X_train, Y_scaled, neigh_size, device=device)

    cems_args = SimpleNamespace(
        sigma       = cfg["sigma"],
        cems_method = cfg["cems_method"],
        id          = d,
        use_hessian = True,
    )
    return cems_args, knn_indices, neigh_size, y_scaler

## Experiment 1: Baseline (real data only)

DINOv2 features from cache → BiomassModel → CEMS training. Reference score.

In [6]:
print("=== EXPERIMENT 1: Baseline (real data only) ===")

cems_args_1, knn_1, neigh_1, scaler_1 = make_cems_args(
    X_real_train, Y_real_train, cfg, device, label="baseline"
)

torch.manual_seed(cfg["seed"])
model_1 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

history_1 = train_cems(
    model=model_1,
    X_train=X_real_train,
    Y_train_raw=Y_real_train,
    val_ds=val_ds,
    knn_indices=knn_1,
    scaler=scaler_1,
    args=cems_args_1,
    neigh_size=neigh_1,
    epochs=cfg["epochs"],
    lr=cfg["lr"],
    weight_decay=cfg["weight_decay"],
    seed=cfg["seed"],
    device=device,
    verbose=True,
)

_, r2_1, rmse_1, _, _ = eval_epoch(model_1, val_loader, device)
print(f"\nBaseline  val R\u00b2={r2_1:.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_1[t]:.4f}")

=== EXPERIMENT 1: Baseline (real data only) ===
  [latent baseline]  d = 9.21  (method: skdim.TwoNN)
  baseline: d=9.21 → d=9  neigh_size=64
  [train_cems] mode=CEMS-full  neigh_size=64  sigma=0.001  epochs=80
  ep   1  tr=7.0460  real=3.5228  aug=3.5232  val_loss=2.1570  val_R²=0.5854  [Green:15.83  Dead:9.39  Clover:15.87  g:14.56  Total:15.62]
  ep  10  tr=1.8523  real=0.9333  aug=0.9190  val_loss=1.5612  val_R²=0.7895  [Green:8.97  Dead:7.82  Clover:12.52  g:10.35  Total:11.79]
  ep  20  tr=1.3566  real=0.6753  aug=0.6812  val_loss=1.6179  val_R²=0.7725  [Green:10.54  Dead:6.97  Clover:12.02  g:10.96  Total:12.47]
  ep  30  tr=1.2154  real=0.6090  aug=0.6064  val_loss=1.6239  val_R²=0.7729  [Green:10.92  Dead:6.95  Clover:12.26  g:11.84  Total:12.63]
  ep  40  tr=1.1176  real=0.5604  aug=0.5573  val_loss=1.7981  val_R²=0.7536  [Green:11.15  Dead:9.14  Clover:12.46  g:11.57  Total:14.33]
  ep  50  tr=1.0654  real=0.5314  aug=0.5340  val_loss=1.8069  val_R²=0.7640  [Green:11.33  Dead

## Experiment 2: RandAugment baseline

Real images re-extracted with RandAugment applied before DINOv2. Comparison to show classical augmentation vs diffusion-based.

In [7]:
print("=== EXPERIMENT 2: RandAugment baseline ===")

_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
_transform_randaug = transforms.Compose([
    transforms.Resize((252, 504)),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(), _norm,
])

print("Loading DINOv2 for RandAugment extraction...")
dino_model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)
dino_model.eval().to(device)
for p in dino_model.parameters():
    p.requires_grad_(False)

train_image_id_list = [image_ids[i] for i in train_idx]
id_to_path = {
    _image_id_from_path(row["image_path"]): REPO_ROOT / "data" / "image" / row["image_path"]
    for _, row in df_wide.iterrows()
}

print(f"Extracting RandAugment features for {len(train_image_id_list)} images...")
ra_feats = []
for i, iid in enumerate(train_image_id_list):
    if i % 50 == 0: print(f"  {i}/{len(train_image_id_list)}")
    p = id_to_path.get(iid)
    if p is None or not p.exists():
        ra_feats.append(np.zeros(384, dtype=np.float32))
        continue
    img = Image.open(p).convert("RGB")
    x = _transform_randaug(img).unsqueeze(0).to(device)
    with torch.no_grad():
        ra_feats.append(dino_model(x).squeeze(0).cpu().numpy())
X_ra_train = np.stack(ra_feats).astype(np.float32)
del dino_model

cems_args_2, knn_2, neigh_2, scaler_2 = make_cems_args(
    X_ra_train, Y_real_train, cfg, device, label="randaugment"
)

torch.manual_seed(cfg["seed"])
model_2 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

history_2 = train_cems(
    model=model_2,
    X_train=X_ra_train,
    Y_train_raw=Y_real_train,
    val_ds=val_ds,
    knn_indices=knn_2,
    scaler=scaler_2,
    args=cems_args_2,
    neigh_size=neigh_2,
    epochs=cfg["epochs"],
    lr=cfg["lr"],
    weight_decay=cfg["weight_decay"],
    seed=cfg["seed"],
    device=device,
    verbose=True,
)

_, r2_2, rmse_2, _, _ = eval_epoch(model_2, val_loader, device)
print(f"\nRandAugment  val R²={r2_2:.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_2[t]:.4f}")

=== EXPERIMENT 2: RandAugment baseline ===
Loading DINOv2 for RandAugment extraction...
Extracting RandAugment features for 286 images...
  0/286
  50/286
  100/286
  150/286
  200/286
  250/286
  [latent randaugment]  d = 11.05  (method: skdim.TwoNN)
  randaugment: d=11.05 → d=11  neigh_size=128
  [train_cems] mode=CEMS-full  neigh_size=128  sigma=0.001  epochs=80
  ep   1  tr=6.9019  real=3.4497  aug=3.4522  val_loss=2.0714  val_R²=0.6116  [Green:14.82  Dead:9.21  Clover:16.57  g:14.81  Total:14.81]
  ep  10  tr=1.6132  real=0.8097  aug=0.8036  val_loss=1.9925  val_R²=0.7177  [Green:11.22  Dead:10.31  Clover:12.96  g:12.98  Total:16.61]
  ep  20  tr=1.2180  real=0.6075  aug=0.6105  val_loss=1.8856  val_R²=0.7281  [Green:12.43  Dead:8.09  Clover:13.04  g:12.66  Total:15.02]
  ep  30  tr=1.0655  real=0.5301  aug=0.5353  val_loss=2.0177  val_R²=0.7223  [Green:14.45  Dead:8.45  Clover:12.31  g:13.35  Total:16.31]
  ep  40  tr=1.0236  real=0.5093  aug=0.5143  val_loss=2.0512  val_R²=0.696

## Experiment 3: DA-Fusion (real + synthetic)

Combines real training features with DINOv2 features from DA-Fusion synthetic images.
kNN is precomputed over the combined training set. Validation is on real images only.

**Requires**: run `generate_augmented.py` then `extract_features.py` to produce the augmented cache.

In [8]:
if not AUG_CACHE_EXISTS or not AUG_CSV_EXISTS:
    print("[SKIP] Augmented cache not available. See warnings above.")
else:
    print("=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===")
    print(f"  Real train:      {len(X_real_train)}")
    print(f"  Synthetic train: {len(aug_features_np)}")

    # Combine real train + synthetic for training
    X_daf_train = np.vstack([X_real_train, aug_features_np])   # (N_train + K, 384)
    Y_daf_train = np.vstack([Y_real_train, aug_labels_np])     # (N_train + K, 5)

    cems_args_3, knn_3, neigh_3, scaler_3 = make_cems_args(
        X_daf_train, Y_daf_train, cfg, device, label="da-fusion"
    )

    torch.manual_seed(cfg["seed"])
    model_3 = BiomassModel(384, cfg["latent_dim"], 5, cfg["dropout"]).to(device)

    history_3 = train_cems(
        model=model_3,
        X_train=X_daf_train,
        Y_train_raw=Y_daf_train,
        val_ds=val_ds,
        knn_indices=knn_3,
        scaler=scaler_3,
        args=cems_args_3,
        neigh_size=neigh_3,
        epochs=cfg["epochs"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        seed=cfg["seed"],
        device=device,
        verbose=True,
    )

    _, r2_3, rmse_3, _, _ = eval_epoch(model_3, val_loader, device)
    print(f"\nDA-Fusion  val R\u00b2={r2_3:.4f}")
    for t in TARGETS:
        print(f"  RMSE {t:<18}: {rmse_3[t]:.4f}")

=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===
  Real train:      286
  Synthetic train: 1071
  [latent da-fusion]  d = 14.02  (method: skdim.TwoNN)
  da-fusion: d=14.02 → d=14  neigh_size=128
  [train_cems] mode=CEMS-full  neigh_size=128  sigma=0.001  epochs=80
  ep   1  tr=5.1809  real=2.5908  aug=2.5900  val_loss=2.4564  val_R²=0.5802  [Green:17.33  Dead:9.12  Clover:13.96  g:15.02  Total:18.39]
  ep  10  tr=1.6172  real=0.8082  aug=0.8090  val_loss=2.3253  val_R²=0.6106  [Green:16.84  Dead:9.80  Clover:12.21  g:15.95  Total:19.23]
  ep  20  tr=1.2634  real=0.6324  aug=0.6310  val_loss=1.6615  val_R²=0.7467  [Green:12.32  Dead:6.80  Clover:11.85  g:12.65  Total:14.26]
  ep  30  tr=1.0925  real=0.5464  aug=0.5461  val_loss=1.9775  val_R²=0.6858  [Green:14.57  Dead:8.03  Clover:12.26  g:14.11  Total:16.50]
  ep  40  tr=1.0137  real=0.5078  aug=0.5059  val_loss=1.8405  val_R²=0.7604  [Green:11.88  Dead:7.80  Clover:10.71  g:12.64  Total:15.32]
  ep  50  tr=0.9563  real=0.4773  aug=

## Results Summary

In [9]:
header = f"{'Method':<16} | {'Val R\u00b2':>6} | " + " | ".join(f"{t.split('_')[1]:>8}" for t in TARGETS)
sep = "-" * len(header)
print(sep)
print(header)
print(sep)

rows = [("Baseline", r2_1, rmse_1), ("RandAugment", r2_2, rmse_2)]
if AUG_CACHE_EXISTS and AUG_CSV_EXISTS:
    rows.append(("DA-Fusion", r2_3, rmse_3))

for name, r2, rmse in rows:
    rmse_vals = " | ".join(f"{rmse[t]:>8.2f}" for t in TARGETS)
    print(f"{name:<16} | {r2:>6.4f} | {rmse_vals}")
print(sep)

--------------------------------------------------------------------------------
Method           | Val R² |    Green |     Dead |   Clover |        g |    Total
--------------------------------------------------------------------------------
Baseline         | 0.7612 |    11.18 |     8.37 |    12.32 |    11.68 |    14.13
RandAugment      | 0.7093 |    14.58 |     8.55 |    12.89 |    13.63 |    16.29
DA-Fusion        | 0.7714 |    10.88 |     7.83 |    10.87 |    11.81 |    14.03
--------------------------------------------------------------------------------
